## NHANES Dataset Analysis

This notebook focuses on how to use both the predictive model and the conformance inference model using the National Health and Nutrition Examination Survey (NHANES) dataset. The dataset can be downloaded at https://wwwn.cdc.gov/nchs/nhanes/. 

The dataset has undergone preprocessing steps which includes removing unnecessary variables as well as missing data. More info in the data directory.


In [9]:
import sys
sys.path.append("..")

import os
import pandas as pd

import random
import numpy as np
import torch
from pprint import pprint

from data import normalize_value
from data_real import load_data as load_data_real
from train import train_conformance_inference
from train_bc import cv_loop_bc
from metrics import compute_classification_metrics
from utils import get_device

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, 
    confusion_matrix, roc_auc_score, roc_curve, 
    precision_recall_curve, auc
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, 
    confusion_matrix, roc_auc_score, log_loss
)

device = get_device()

random.seed(0)
torch.manual_seed(0)
np.random.seed(0)

We initialize the main parameters used during the training of the predictive model. The model is training using a `n_fold` cross-validation approach. The number of epochs, batch size, learning rate, weight decay are set to 1000, 64, 0.001, and 0.01, respectively. These values ought to be adjusted in accordance with the specific requirements of the problem, potentially through the application of a grid search technique.

The classification model is trained with an early stop of 4 and no delta. The regression model uses an early stop of 6 with a no delta.


In [10]:
n_folds = 5
n_epochs = 300
batch_size = 64
learning_rate = 0.001
weight_decay = 0.01
dropout = 0.2
hidden_sizes = [32, 16, 8]
patience_classification = 20
min_delta_classification = 0.0
patience_regression = 4
min_delta_regression = 0.0
normalize = False
standardize = True

In [11]:
def evaluate_on_test(model, test_data):
    """Evalúa el modelo en el conjunto de test y retorna métricas detalladas"""
    model.eval()
    with torch.no_grad():
        y_pred_proba = model(
            torch.from_numpy(test_data['x']).to(device)
        ).cpu().detach().numpy()
    
    y_pred_bin = (y_pred_proba >= 0.5).astype(int)
    y_true = np.ravel(test_data['y'])
    
    # Métricas básicas
    accuracy = accuracy_score(y_true, y_pred_bin)
    auc_score = roc_auc_score(y_true, y_pred_proba)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_bin, average='weighted'
    )
    cm = confusion_matrix(y_true, y_pred_bin)
    
    # Cross-entropy (log loss)
    ce = log_loss(y_true, y_pred_proba)
    
    return {
        'accuracy': accuracy,
        'auc': auc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'ce': ce,
        'cm': cm,
    }

In [12]:
comb_1 = ['RIDAGEYR', 'RIAGENDR']
comb_2 = ['BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS']
comb_3 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS']
comb_4 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR']
comb_5 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGH']
comb_6 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGLU']
comb_7 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGLU', 'LBXGH']
combinations = [comb_1, comb_2, comb_3, comb_4, comb_5, comb_6, comb_7]

# Load pre-generated splits (run generate_splits.py once to create this file)
splits_data = np.load("../data/splits.npz")
train_seqn = splits_data['train_seqn']
test_seqn  = splits_data['test_seqn']

results_list = []

for i, comb in enumerate(combinations):
    print(f"[{i+1}/{len(combinations)}] Procesando combinación {i}: {len(comb)} features")

    file_real = os.path.join("../data", "nhanes_with_metabolic_syndrome_adults.csv")
    data = load_data_real(file_real, comb, 'metabolic_syndrome', 'WTMEC2YR')

    # Map pre-generated SEQN splits to positional indices in this combination's data
    seqn = data['seqn']
    train_split = np.where(np.isin(seqn, train_seqn))[0]
    test_split  = np.where(np.isin(seqn, test_seqn))[0]

    # Build fold splits as indices into the train subset
    train_seqn_local = seqn[train_split]
    fold_splits = []
    for fold_idx in range(n_folds):
        ft = splits_data[f'fold_{fold_idx}_train']
        fv = splits_data[f'fold_{fold_idx}_val']
        fold_splits.append((
            np.where(np.isin(train_seqn_local, ft))[0],
            np.where(np.isin(train_seqn_local, fv))[0],
        ))

    train_data = {
        'seqn': data['seqn'][train_split],
        'x': data['x'][train_split],
        'y': data['y'][train_split],
        'w': data['w'][train_split],
        'z': data['z'][train_split]
    }

    test_data = {
        'seqn': data['seqn'][test_split],
        'x': data['x'][test_split],
        'o': data['x'][test_split],
        'y': data['y'][test_split],
        'w': data['w'][test_split],
        'z': data['z'][test_split]
    }

    if standardize:
        scaler = StandardScaler()
        train_data['x'] = scaler.fit_transform(train_data['x'])
        test_data['x'] = scaler.transform(test_data['x'])

    model, train_metrics = cv_loop_bc(
        data=train_data,
        splits=fold_splits,
        n_epochs=n_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        patience=patience_classification,
        min_delta=min_delta_classification,
        dropout=dropout,
        hidden_sizes=hidden_sizes
    )

    test_metrics = evaluate_on_test(model, test_data)

    result_row = {
        'Combination': i,
        'N_Features': len(comb),
        'Features': ', '.join(comb),
        'Train_AUC': float(train_metrics['auc']),
        'Train_Accuracy': float(train_metrics['accuracy']),
        'Train_Precision': float(train_metrics['precision']),
        'Train_Recall': float(train_metrics['recall']),
        'Train_F1': float(train_metrics['f1']),
        'Train_CE': float(train_metrics['ce']),
        'Test_AUC': float(test_metrics['auc']),
        'Test_Accuracy': float(test_metrics['accuracy']),
        'Test_Precision': float(test_metrics['precision']),
        'Test_Recall': float(test_metrics['recall']),
        'Test_F1': float(test_metrics['f1']),
        'Test_CE': float(test_metrics['ce'])
    }
    results_list.append(result_row)

    print(f"  Train - AUC: {train_metrics['auc']:.4f}, Acc: {train_metrics['accuracy']:.4f}, F1: {train_metrics['f1']:.4f}, CE: {train_metrics['ce']:.4f}")
    print(f"  Test  - AUC: {test_metrics['auc']:.4f}, Acc: {test_metrics['accuracy']:.4f}, F1: {test_metrics['f1']:.4f}, CE: {test_metrics['ce']:.4f}")
    print()

results_df = pd.DataFrame(results_list)

print("\n" + "="*120)
print("RESULTADOS COMPARATIVOS")
print("="*120)
display(results_df[[
    'Combination', 'N_Features',
    'Train_AUC', 'Train_Accuracy', 'Train_F1', 'Train_CE',
    'Test_AUC', 'Test_Accuracy', 'Test_F1', 'Test_CE'
]].style.format({
    'Train_AUC': '{:.4f}', 'Train_Accuracy': '{:.4f}',
    'Train_F1': '{:.4f}', 'Train_CE': '{:.4f}',
    'Test_AUC': '{:.4f}', 'Test_Accuracy': '{:.4f}',
    'Test_F1': '{:.4f}', 'Test_CE': '{:.4f}'
}).background_gradient(subset=['Test_AUC'], cmap='RdYlGn'))

results_df.to_csv('comparison_results_nn_2.csv', index=False)
print(f"\nResultados guardados en 'comparison_results_nn_2.csv'")


[1/7] Procesando combinación 0: 2 features
  Early stopping at epoch 74
  Early stopping at epoch 31
  Early stopping at epoch 27
  Early stopping at epoch 45
  Early stopping at epoch 28
  Train - AUC: 0.6973, Acc: 0.6725, F1: 0.3952, CE: 3.8570
  Test  - AUC: 0.6827, Acc: 0.6691, F1: 0.6506, CE: 0.5905

[2/7] Procesando combinación 1: 7 features
  Early stopping at epoch 49
  Early stopping at epoch 45
  Early stopping at epoch 69
  Early stopping at epoch 45
  Early stopping at epoch 90
  Train - AUC: 0.8677, Acc: 0.7996, F1: 0.6788, CE: 2.0405
  Test  - AUC: 0.8638, Acc: 0.7857, F1: 0.7798, CE: 0.4387

[3/7] Procesando combinación 2: 9 features
  Early stopping at epoch 62
  Early stopping at epoch 72
  Early stopping at epoch 93
  Early stopping at epoch 55
  Early stopping at epoch 82
  Train - AUC: 0.8728, Acc: 0.8042, F1: 0.6702, CE: 2.2534
  Test  - AUC: 0.8687, Acc: 0.7929, F1: 0.7790, CE: 0.4269

[4/7] Procesando combinación 3: 11 features
  Early stopping at epoch 113
  Ear

,Combination,N_Features,Train_AUC,Train_Accuracy,Train_F1,Train_CE,Test_AUC,Test_Accuracy,Test_F1,Test_CE
0,0,2,0.6973,0.6725,0.3952,3.8570,0.6827,0.6691,0.6506,0.5905
1,1,7,0.8677,0.7996,0.6788,2.0405,0.8638,0.7857,0.7798,0.4387
2,2,9,0.8728,0.8042,0.6702,2.2534,0.8687,0.7929,0.7790,0.4269
3,3,11,0.9359,0.8637,0.7841,1.3159,0.9364,0.8589,0.8569,0.3152
4,4,12,0.9375,0.8648,0.7895,1.3351,0.9413,0.8725,0.8714,0.3031
5,5,12,0.9595,0.8943,0.8317,1.0792,0.9601,0.8945,0.8941,0.2510
6,6,13,0.9592,0.8951,0.8382,1.0631,0.9577,0.8889,0.8887,0.2561



Resultados guardados en 'comparison_results_nn_2.csv'


In this instance, we employ a distinct combination of input variables, with the diabetes variable serving as the class variable. The NHANES dataset is partitioned into training and test sets in a 75/25 ratio. 